# 11 — Calibration and θ: why the reasoner knows when to shut up


> **Note.** The θ-calibration argument here is cited by the README and applies to the cassette-native reasoner too — `store.ask(Query)` and `store.connect()` return the same Dempster–Shafer mass, with θ → 1.0 on claims the corpus doesn't answer. The code below runs against the legacy `InfonEngine` class; for the cassette version see **[03 — Querying](03_querying.ipynb)**, section 'Why θ matters'.

A single number, θ, is the difference between "a confident hallucination"
and "a trustworthy I don't know." This notebook is the full story of
that number on a real corpus:

1. We'll build a reasoner without the relevance filter, show that it
   hallucinates confidently on claims the corpus doesn't speak to
   (θ ≈ 0.00 on NEI).
2. We'll turn on the strict role-wise filter, show θ jumps to 1.00 on
   the same NEI claims.
3. We'll show that SUPPORTS claims keep a committed θ around 0.25.
4. We'll probe the three honest edge cases: partial-role matches,
   contradictory evidence, active exploration via `cog.expand`.

The single experimental result: **NEI θ 0.00 → 1.00, accuracy 45% → 85%**
on the 20-claim LLM comparison gold set. Every other calibration
invariant in the library depends on this one mechanism working.


## 1. The shared EV-battery corpus

Same 12 documents as `10_automl.ipynb` and `09_self_supervised_mdp.ipynb`
so you can cross-reference verdicts across the three notebooks.


In [ ]:
import json, os, tempfile
from infon import InfonEngine, InfonConfig

SCHEMA = {
    "toyota":   {"type": "actor",    "tokens": ["toyota"]},
    "honda":    {"type": "actor",    "tokens": ["honda"]},
    "tesla":    {"type": "actor",    "tokens": ["tesla"]},
    "panasonic":{"type": "actor",    "tokens": ["panasonic"]},
    "catl":     {"type": "actor",    "tokens": ["catl"]},
    "invests":  {"type": "relation", "tokens": ["invest", "invests", "investment"]},
    "partners": {"type": "relation", "tokens": ["partner", "partners", "partnership"]},
    "produces": {"type": "relation", "tokens": ["produce", "produces", "produced"]},
    "expands":  {"type": "relation", "tokens": ["expand", "expands", "expansion"]},
    "delays":   {"type": "relation", "tokens": ["delay", "delays", "delayed"]},
    "acquires": {"type": "relation", "tokens": ["acquire", "acquires", "acquired"]},
    "battery":  {"type": "feature",  "tokens": ["battery", "batteries"]},
    "factory":  {"type": "feature",  "tokens": ["factory", "plant"]},
    "ev":       {"type": "feature",  "tokens": ["ev", "electric vehicle"]},
    "japan":    {"type": "market",   "tokens": ["japan", "japanese"]},
    "china":    {"type": "market",   "tokens": ["china", "chinese"]},
    "na":       {"type": "market",   "tokens": ["north america", "united states"]},
}

DOCS = [
    {"id": "d01", "timestamp": "2024-01-10", "text": "Toyota invests in battery technology in Japan."},
    {"id": "d02", "timestamp": "2024-02-15", "text": "Toyota partners with Panasonic on battery development."},
    {"id": "d03", "timestamp": "2024-03-22", "text": "Toyota produces prototype batteries."},
    {"id": "d04", "timestamp": "2024-01-05", "text": "Tesla expands its battery factory in North America."},
    {"id": "d05", "timestamp": "2024-02-28", "text": "Tesla produces batteries at its Gigafactory."},
    {"id": "d06", "timestamp": "2024-04-01", "text": "Tesla acquires battery supply chain assets."},
    {"id": "d07", "timestamp": "2024-01-20", "text": "Honda partners with CATL on battery supply in China."},
    {"id": "d08", "timestamp": "2024-02-10", "text": "Honda delays its EV production timeline."},
    {"id": "d09", "timestamp": "2024-03-15", "text": "Honda invests in battery research."},
    {"id": "d10", "timestamp": "2024-01-25", "text": "CATL expands battery production in China."},
    {"id": "d11", "timestamp": "2024-02-20", "text": "CATL produces batteries for Japanese automakers."},
    {"id": "d12", "timestamp": "2024-01-30", "text": "Panasonic invests in battery factory in Japan."},
]

tmpdir = tempfile.mkdtemp(prefix="calibration-")
schema_path = os.path.join(tmpdir, "schema.json")
with open(schema_path, "w") as f:
    json.dump(SCHEMA, f)

cog = InfonEngine(InfonConfig(
    schema_path=schema_path,
    db_path=os.path.join(tmpdir, "cog.db"),
    quality_threshold=0.05,
    max_triples_per_sentence=2,
    random_state=42,
))
for d in DOCS:
    cog.ingest([d])
cog.consolidate()
print(f"ingested {cog.stats()['infon_count']} infons")


## 2. Why θ exists — the fourth number problem

Traditional classifiers output a probability distribution over a label
set. If your gold labels are `{SUPPORTS, REFUTES, NOT_ENOUGH_INFO}` and
the model has no evidence, softmax still forces a pick — *usually the
most frequent label in training.*

Dempster–Shafer mass theory adds a fourth number, `θ`, that absorbs
unassignable evidence. A mass function is `(supports, refutes,
uncertain, θ)` summing to 1:

| Channel | Meaning |
|---|---|
| **supports** | mass on {SUPPORTS} |
| **refutes**  | mass on {REFUTES} |
| **uncertain** | mass on {ambiguous — evidence is weak or mixed} |
| **θ** | mass on the whole frame — "I genuinely don't know" |

The key property: a mass with `θ = 1.0` is the *neutral element* for
Dempster combination. Combining "I know A" with "I don't know B" gives
back "I know A" — ignorance doesn't undo knowledge. That's what makes
θ suitable as a *gate* rather than a guess.


## 3. The calibration bug we almost shipped

Before the relevance filter, the reasoner ran Dempster's rule over
*every infon with any anchor overlap*. That sounds reasonable — but
Dempster's rule has an underappreciated property: **combining many
weakly-relevant masses concentrates mass fast**, even when no single
infon deserved confidence.

Let's reproduce the failure mode by pulling every infon into a single
verdict via the raw DS combiner — no relevance filter, no role-wise
overlap check.


In [ ]:
from infon.dempster_shafer import (
    MassFunction, combine_multiple,
    mass_from_polarity, mass_from_confidence,
)

def reason_without_filter(cog, query: str):
    '''Naive: combine DS masses from every infon in the store.'''
    infons = cog.store.query_infons(limit=200)
    if not infons:
        return MassFunction(theta=1.0)
    # For each infon, fuse two quick DS sources
    masses = []
    for inf in infons:
        m = combine_multiple([
            mass_from_polarity(inf),
            mass_from_confidence(inf),
        ])
        masses.append(m)
    return combine_multiple(masses)

# Try two queries — one SUPPORTS, one NEI
for q in ["Did Toyota invest in batteries?",
          "Did Tesla acquire CATL?"]:
    m = reason_without_filter(cog, q)
    print(f"  {q:45s}")
    print(f"    supports={m.supports:.2f}  refutes={m.refutes:.2f}  "
          f"uncertain={m.uncertain:.2f}  θ={m.theta:.2f}")
    print()


**Both queries get the same mass function** — because we're running
Dempster over the whole corpus regardless of what was asked. θ has
concentrated toward 0.00 for both, even the one that the corpus says
nothing about. The system is hallucinating.

That's the trap: Dempster's rule converges *because* ignorance combines
away. It doesn't know what's relevant to the *query*; it just has facts.


## 4. The fix: strict role-wise relevance filter

Inside `HypergraphReasoner.reason()`, before Dempster's rule is applied,
each candidate infon is scored on role-wise overlap with the query:

```python
subj_s = query_activations.get(infon.subject, 0.0)
pred_s = query_activations.get(infon.predicate, 0.0)
obj_s  = query_activations.get(infon.object, 0.0)

min_role = min(subj_s, pred_s, obj_s)
if min_role <= 0.05:
    continue    # any role scoring near zero → drop

rel = (subj_s * pred_s * obj_s) ** (1/3)   # geometric mean
```

The rule is: **all three roles of the infon must activate on the query
for the infon to count as evidence.** A single missing role sinks the
whole infon.

On NEI queries, no infon passes the filter, so Dempster's rule is never
invoked, and `reason()` returns the uninformative prior `(0, 0, 0, 1)`.
That's the whole fix.


In [ ]:
reasoner = cog.reasoner()

cases = [
    ("Did Toyota invest in batteries?",       "SUPPORTS"),
    ("Did Tesla produce batteries?",          "SUPPORTS"),
    ("Did Honda partner with CATL?",          "SUPPORTS"),
    ("Did Panasonic invest in batteries?",    "SUPPORTS"),

    ("Did Tesla acquire CATL?",               "NOT ENOUGH INFO"),
    ("Did Honda produce batteries in Japan?", "NOT ENOUGH INFO"),
    ("Did Toyota merge with Honda?",          "NOT ENOUGH INFO"),
    ("Did BMW invest in batteries?",          "NOT ENOUGH INFO"),
]

rows = []
for q, expected in cases:
    r = reasoner.reason(q)
    rows.append({
        "query": q,
        "gold": expected,
        "verdict": r.verdict,
        "S": r.mass.supports,
        "R": r.mass.refutes,
        "U": r.mass.uncertain,
        "theta": r.mass.theta,
        "correct": r.verdict == expected,
    })

# Print as a compact table
print(f"{'query':<45s} {'gold':<16s} {'got':<16s} "
      f"{'S':>5s} {'R':>5s} {'U':>5s} {'θ':>5s}  ok")
print("-" * 100)
for row in rows:
    tick = "✓" if row["correct"] else "✗"
    print(f"{row['query']:<45s} "
          f"{row['gold']:<16s} "
          f"{row['verdict']:<16s} "
          f"{row['S']:>5.2f} {row['R']:>5.2f} "
          f"{row['U']:>5.2f} {row['theta']:>5.2f}  {tick}")


## 5. The bimodality that carries the story

Look at the θ column. SUPPORTS claims have low θ (committed); NEI
claims have θ close to 1.0 (honest refusal). **The reasoner is
bimodal in θ** — it's either confident or ignorant, never stuck in the
middle.

Let's plot it to make the pattern unmistakable.


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
colors = ["#4a9eff" if r["gold"] == "SUPPORTS" else "#ff7050" for r in rows]
labels = [r["query"][:26] + "…" for r in rows]
thetas = [r["theta"] for r in rows]

ax.barh(range(len(rows)), thetas, color=colors, alpha=0.85)
ax.set_yticks(range(len(rows)))
ax.set_yticklabels(labels, fontsize=8)
ax.set_xlim(0, 1.05)
ax.axvline(0.5, color="grey", linestyle="--", alpha=0.5, linewidth=1)
ax.set_xlabel("θ (residual ignorance)")
ax.set_title("θ bimodality: SUPPORTS vs NEI on the EV corpus")
ax.legend(handles=[
    plt.Rectangle((0,0),1,1, color="#4a9eff", alpha=0.85, label="SUPPORTS gold"),
    plt.Rectangle((0,0),1,1, color="#ff7050", alpha=0.85, label="NEI gold"),
], loc="lower right")
ax.invert_yaxis()
plt.tight_layout()
plt.show()


The gap between the blue bars (SUPPORTS) and the orange bars (NEI) is
the whole calibration story. A good reasoner *shouldn't* have bars
clustering around 0.5 — that would mean it was equivocating on claims
where it actually does / doesn't have evidence.


## 6. Three edge cases worth understanding

Calibration isn't a single threshold — it's a *behaviour*. Three
situations worth probing before trusting θ in production.


### Edge case 1: partial role overlap

What about a claim where the subject and predicate activate but the
object is wrong? Example: "Did Toyota invest in televisions?" — Toyota
activates, invest activates, television doesn't match any anchor. The
relevance filter should drop every infon (because the object role has
near-zero overlap), and θ should go to 1.0.


In [ ]:
r = reasoner.reason("Did Toyota invest in televisions?")
print(f"verdict: {r.verdict}")
print(f"mass:    S={r.mass.supports:.2f}  θ={r.mass.theta:.2f}")
print(f"\n→ A missing-object claim is held to the same standard "
      f"as a fully-missing claim.")


### Edge case 2: contradictory evidence

What if the corpus contains both "X partners with Y" and "X competes
with Y"? Neither infon is wrong; they're just in tension. Dempster's
rule handles this via the conflict term K — mass that can't be
assigned to any consistent subset gets normalized away, raising θ
as a signal of tension.

Our corpus doesn't contain a direct contradiction, so let's simulate
by asking a claim that cuts across evidence in opposite directions:


In [ ]:
# Honda partners with CATL (d07) but also "Honda invests in battery research" (d09)
# — ask whether Honda invests in CATL. No direct support, subject activates,
# predicate activates, object activates — but no single infon says yes.
r = reasoner.reason("Did Honda invest in CATL?")
print(f"verdict:    {r.verdict}")
print(f"mass:       S={r.mass.supports:.2f}  R={r.mass.refutes:.2f}  "
      f"U={r.mass.uncertain:.2f}  θ={r.mass.theta:.2f}")


### Edge case 3: active exploration

When θ is high *and* you can fetch more evidence, `cog.expand()` is the
right response. It does three things in sequence: fetch candidate
documents from a search backend, ingest them, re-query. The before/after
θ difference tells you whether the exploration helped.

We'll use a mock search backend here so the notebook runs offline.
Swap `source="mock"` for `source="ddgs"` after `pip install ddgs` to use
the real meta-search.


In [ ]:
def mock_search(query):
    # Pretend ddgs found a supporting snippet
    return [{
        "title":  "Tesla and CATL finalize deal",
        "body":   "Tesla announced the acquisition of CATL battery assets for $10B.",
        "href":   "https://example.com/tesla-catl",
    }]

result = cog.expand(
    "Did Tesla acquire CATL?",
    theta_threshold=0.3,
    source="mock",
    search_fn=mock_search,
    verbose=True,
)

print()
print(f"expanded:  {result['expanded']}")
print(f"n_new_docs: {result['n_new_docs']}")
print(f"θ before:  {result['before']['theta']:.3f}")
print(f"θ after:   {result['after']['theta']:.3f}  ← lower")


Notice what this buys you: **the decision to go fetch more evidence is
principled**. You don't fetch unless θ is high; you don't overwrite
existing knowledge unless the new evidence earns it. That's
retrieval-augmented generation with a gate instead of a bulk-add.


## 7. Checkpoint tests

The library ships these as first-class test gates. If they ever break,
calibration is broken — no release goes out without them passing.
`tests/test_theta_calibration.py` is the file.

| Test | Gate |
|---|---|
| `test_nei_claims_have_high_theta` | mean θ on NEI > 0.5 on the 20-claim gold set |
| `test_supported_claims_have_low_theta` | mean θ on confident-SUPPORTS < 0.3 |
| `test_accuracy_still_reasonable` | overall accuracy ≥ 0.6 (guards against "perfect calibration, useless accuracy") |

Those three tests together lock the invariant: the system has to be
*honest* (high θ on NEI), *committed* (low θ on SUPPORTS), and *useful*
(high accuracy overall). Any two of the three is a degenerate fix.


## Recap

1. θ is the fourth number that Dempster-Shafer theory adds on top of a
   probability distribution. It absorbs unassignable evidence — "I
   don't know."
2. Naively running Dempster over the whole corpus hallucinates because
   ignorance combines away across many weak sources.
3. The fix is a **strict role-wise relevance filter** *before* combine.
   All three roles of an infon must activate on the query, or the
   infon doesn't count as evidence.
4. Result: NEI θ ≈ 1.00, SUPPORTS θ ≈ 0.25, accuracy 45% → 85%.
5. θ is a gate, not a score — it drives `cog.expand()`, ensemble
   combining, and the production test suite.

**Next:**
- **`09_self_supervised_mdp.ipynb`** — the SSL research arc that
  replaces the six hand-coded DS sources with JEPA prediction error.
- **`10_automl.ipynb`** — the capstone that uses calibrated θ as the
  selection metric across SSL modes.
- `infon-workshop/12-dempster-shafer.md` — the prose module that
  derives the filter from first principles.


In [ ]:
cog.close()
